In [17]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to the same database used for the demand model
conn = sqlite3.connect('/Users/manushdesai/Desktop/multi-agent/data/ecommerce.db')

# We JOIN returns with orders (to get order value), products (to get category),
# and customers (to get the persisted is_repeat_offender flag) --
# these give us the real signals for judging return risk
returns_df = pd.read_sql("""
    SELECT r.*, o.total_amount, p.category, c.is_repeat_offender
    FROM returns r
    JOIN orders o ON r.order_id = o.order_id
    JOIN products p ON r.product_id = p.product_id
    JOIN customers c ON r.customer_id = c.customer_id
""", conn)

print("Total return records:", len(returns_df))
print("Flagged (suspicious):", returns_df['is_flagged'].sum())
print("Not flagged:", len(returns_df) - returns_df['is_flagged'].sum())

returns_df.head()

Total return records: 587
Flagged (suspicious): 203
Not flagged: 384


,return_id,order_id,customer_id,product_id,return_date,reason,days_since_order,customer_total_returns,is_flagged,agent_decision,total_amount,category,is_repeat_offender
0,1,2,125,43,2025-12-01,Wrong size / doesn't fit,1,1,0,approved,9964.32,Sports,0
1,2,16,96,19,2025-08-13,Wrong size / doesn't fit,8,1,0,approved,5138.12,Sports,0
2,3,19,114,80,2026-03-24,Received wrong item,29,1,0,approved,4136.64,Electronics,0
3,4,29,62,2,2026-05-01,Item not as described,37,1,0,approved,1805.34,Apparel,0
4,5,35,108,48,2026-05-29,Item not as described,15,1,0,approved,6100.20,Apparel,0


In [18]:
# Add this mapping near the top of Cell 2
CATEGORY_RETURN_WINDOW = {
    "Electronics": 7, "Apparel": 15, "Home & Kitchen": 7, "Beauty": 7, "Sports": 7,
}

# NEW: days_past_window -- captures policy violation severity directly
returns_df['return_window'] = returns_df['category'].map(CATEGORY_RETURN_WINDOW).fillna(7)
returns_df['days_past_window'] = (returns_df['days_since_order'] - returns_df['return_window']).clip(lower=0)

# Now one-hot encode BOTH category and reason (previously only category)
returns_df = pd.get_dummies(returns_df, columns=['category', 'reason'], drop_first=True)

# Updated feature list -- add days_past_window, and add reason columns
feature_cols = (
    ['days_since_order', 'days_past_window', 'customer_total_returns', 'total_amount', 'is_repeat_offender']
    + [c for c in returns_df.columns if c.startswith('category_')]
    + [c for c in returns_df.columns if c.startswith('reason_')]
)

X = returns_df[feature_cols]
y = returns_df['is_flagged']

print("Features used:", feature_cols)
print("Feature matrix shape:", X.shape)

Features used: ['days_since_order', 'days_past_window', 'customer_total_returns', 'total_amount', 'is_repeat_offender', 'category_Beauty', 'category_Electronics', 'category_Home & Kitchen', 'category_Sports', 'reason_Found cheaper elsewhere', 'reason_Item damaged on arrival', 'reason_Item not as described', 'reason_Quality not as expected', 'reason_Received wrong item', "reason_Wrong size / doesn't fit"]
Feature matrix shape: (587, 15)


In [19]:
from sklearn.model_selection import train_test_split

# stratify=y keeps the same flagged/not-flagged RATIO in both train and test sets.
# This matters more here than for the demand model, because with ~588 rows
# a random split could accidentally put too few (or too many) flagged cases
# in the test set, making evaluation unreliable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train rows:", len(X_train), " | Flagged in train:", y_train.sum())
print("Test rows:", len(X_test), " | Flagged in test:", y_test.sum())

Train rows: 440  | Flagged in train: 152
Test rows: 147  | Flagged in test: 51


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Baseline: Logistic Regression. class_weight='balanced' tells it to pay
# more attention to the minority (flagged) class during training, since
# otherwise it could get lazy and just predict "not flagged" for everything.
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train, y_train)
lr_pred = log_reg.predict(X_test)

# Our real model: Random Forest Classifier.
# max_depth kept shallow (5) -- with only ~440 training rows, a deep tree
# risks memorizing individual cases instead of learning general patterns.
rf_clf = RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42, class_weight='balanced')
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

print("Both models trained.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Both models trained.


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def evaluate(name, y_true, y_pred):
    # We use precision/recall instead of plain accuracy because flagged
    # cases are the MINORITY class -- a model that just predicts "not
    # flagged" every time would still get high accuracy while being useless.
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"{name:25s}  Precision: {precision:.2f}   Recall: {recall:.2f}   F1: {f1:.2f}")

print("--- Performance on TEST set ---")
evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)

print("\n--- Full classification report (Random Forest) ---")
print(classification_report(y_test, rf_pred, target_names=['Not flagged', 'Flagged']))

--- Performance on TEST set ---
Logistic Regression        Precision: 0.65   Recall: 0.61   F1: 0.63
Random Forest              Precision: 0.63   Recall: 0.61   F1: 0.62

--- Full classification report (Random Forest) ---
              precision    recall  f1-score   support

 Not flagged       0.80      0.81      0.80        96
     Flagged       0.63      0.61      0.62        51

    accuracy                           0.74       147
   macro avg       0.71      0.71      0.71       147
weighted avg       0.74      0.74      0.74       147



In [22]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# With ~588 rows, a single train/test split can be a bit unstable --
# a different random split could give noticeably different numbers.
# 5-fold cross-validation trains/tests 5 times on different slices and
# averages the result, giving a more reliable performance estimate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_precision = cross_val_score(rf_clf, X, y, cv=cv, scoring='precision')
cv_recall = cross_val_score(rf_clf, X, y, cv=cv, scoring='recall')

print("Cross-validated Precision (5 folds):", np.round(cv_precision, 2))
print("Mean Precision:", round(cv_precision.mean(), 2))
print()
print("Cross-validated Recall (5 folds):", np.round(cv_recall, 2))
print("Mean Recall:", round(cv_recall.mean(), 2))

Cross-validated Precision (5 folds): [0.64 0.62 0.69 0.56 0.55]
Mean Precision: 0.61

Cross-validated Recall (5 folds): [0.68 0.59 0.68 0.68 0.59]
Mean Recall: 0.64


In [23]:
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("--- Feature importance (Random Forest Classifier) ---")
print(importances)

--- Feature importance (Random Forest Classifier) ---
customer_total_returns             0.340563
total_amount                       0.132262
days_since_order                   0.131200
is_repeat_offender                 0.119827
days_past_window                   0.101298
category_Sports                    0.028607
reason_Found cheaper elsewhere     0.028037
reason_Wrong size / doesn't fit    0.017491
reason_Quality not as expected     0.017075
category_Beauty                    0.016690
category_Home & Kitchen            0.015878
reason_Received wrong item         0.015003
reason_Item damaged on arrival     0.014441
category_Electronics               0.012263
reason_Item not as described       0.009365
dtype: float64


In [24]:
import joblib
import os

save_dir = '/Users/manushdesai/Desktop/multi-agent/models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(rf_clf, f'{save_dir}/returns_model.pkl')

with open(f'{save_dir}/returns_model_features.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print("Model saved to:", save_dir)

Model saved to: /Users/manushdesai/Desktop/multi-agent/models
